In [8]:
####################################################
#### Iris Gold-Standard-Skript    ###############
####################################################

import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
import joblib

# --- SCHRITT 1: Daten laden.
df = pd.read_csv("Iris.csv")  # Pfad ggf. anpassen
y = df["Species"]
X = df.drop(columns=["Species","Id"])

# --- SCHRITT 2: Der "Goldene Schnitt" (Train-Test-Split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- SCHRITT 3: Die Pipeline bauen ---
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# --- SCHRITT 4: K-Fold Cross Validation (Training & Validierung)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
print("\n--- Cross-Validation Ergebnisse ---")
print(f"Genauigkeit in den Folds: {scores}")
print(f"Durchschnittliche Genauigkeit: {np.mean(scores):.2%} (+/- {np.std(scores):.2%})")

# --- SCHRITT 5: Das finale Modell trainieren ---
pipeline.fit(X_train, y_train)

# --- SCHRITT 6: Finale Evaluation (Die Stunde der Wahrheit) ---
final_score = pipeline.score(X_test, y_test)
print(f"\nFinale Genauigkeit auf dem Test-Set: {final_score:.2%}")

# --- SCHRITT 7: Modell speichern (Deployment) ---
joblib.dump(pipeline, 'meinModell.pkl')
print("\nModell wurde als 'meinModell.pkl' gespeichert.")


--- Cross-Validation Ergebnisse ---
Genauigkeit in den Folds: [0.975 0.95  0.95 ]
Durchschnittliche Genauigkeit: 95.83% (+/- 1.18%)

Finale Genauigkeit auf dem Test-Set: 93.33%

Modell wurde als 'meinModell.pkl' gespeichert.


In [1]:
pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 2.3 MB/s  0:00:10 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 861.5/861.5 kB 9.5 MB/s  0:00:00
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.1
    Uninstalling typing-inspection-0.4.1:
      Successfully uninstalled typing-inspection-0.4.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21/21 [gradio]20/21 [gradio]te]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.7 requires thinc<8.4.0,>=8.3.4, but you have thinc 9.1.1 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gradio as gr
import joblib
import numpy as np

# Modell laden
model = joblib.load("meinModell.pkl")

feature_names = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm",
]

def predict(SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm):
    # Eingaben in der gleichen Reihenfolge wie beim Training
    x = np.array([[SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm]])
    pred = model.predict(x)[0]
    proba = None
    try:
        proba = model.predict_proba(x)[0]
    except Exception:
        pass

    result = f"Predicted Quality: {pred}"
    if proba is not None:
        # Annahme: Klassen sind z.B. ['bad', 'good']
        class_probs = {str(cls): float(p) for cls, p in zip(model.classes_, proba)}
        result += f"\nProbabilities: {class_probs}"
    return result

inputs = [
    gr.Number(label="SepalLengthCm"),
    gr.Number(label="SepalWidthCm"),
    gr.Number(label="PetalLengthCm"),
    gr.Number(label="PetalWidthCm"),
]

demo = gr.Interface(
    fn=predict,
    inputs=inputs,
    outputs=gr.Textbox(label="Prediction"),
    title="Qualitätsvorhersage",
    description=(
        "LogisticRegression-Modell zur Vorhersage von 'IRIS'"
    ),
)

if __name__ == "__main__":
    #demo.launch()
    #demo.launch(share=True)
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False,
        auth=("student", "geheim")
    )


ERROR:    [Errno 48] error while attempting to bind on address ('0.0.0.0', 7860): [errno 48] address already in use


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.